In [ ]:
# Colab / 新環境第一次執行時，取消下一行註解安裝套件。
# %pip install -q pypdf python-docx pandas openai python-dotenv

from io import BytesIO
import json
import os
import re

import pandas as pd
from docx import Document
from pypdf import PdfReader

print("Week 9 文件處理環境載入完成 ✅")

In [ ]:
def decode_text_bytes(data: bytes) -> str:
    """依常見編碼順序解碼純文字，全部失敗時給出明確錯誤。"""
    for encoding in ("utf-8-sig", "utf-8", "cp950"):
        try:
            return data.decode(encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError("無法辨識文字編碼，請將檔案另存為 UTF-8 後再試。")


sample_bytes = "第一段：文件處理。\n第二段：準備 chunking。".encode("utf-8")
print(decode_text_bytes(sample_bytes))

In [ ]:
def extract_pdf_text(data: bytes) -> str:
    """逐頁抽取 PDF 文字，保留頁碼標記以利後續追蹤來源。"""
    reader = PdfReader(BytesIO(data))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        if page_text.strip():
            pages.append(f"[第 {page_number} 頁]\n{page_text.strip()}")
    if not pages:
        raise ValueError("PDF 沒有可抽取的文字，它可能是掃描檔，需要 OCR。")
    return "\n\n".join(pages)


print("extract_pdf_text 已定義（需有 PDF 位元組才能實際執行）")

In [ ]:
def extract_docx_text(data: bytes) -> str:
    """讀取 Word 段落與表格，將內容轉成統一文字。"""
    document = Document(BytesIO(data))
    blocks = [p.text.strip() for p in document.paragraphs if p.text.strip()]
    for table_index, table in enumerate(document.tables, start=1):
        blocks.append(f"[表格 {table_index}]")
        for row in table.rows:
            values = [cell.text.strip() for cell in row.cells]
            blocks.append(" | ".join(values))
    if not blocks:
        raise ValueError("Word 文件沒有可讀取的段落或表格文字。")
    return "\n".join(blocks)


print("extract_docx_text 已定義")

In [ ]:
def extract_csv_text(data: bytes, max_rows: int = 200) -> str:
    """保留欄名，將 CSV 前 max_rows 列轉成適合摘要的文字。"""
    text = decode_text_bytes(data)
    frame = pd.read_csv(BytesIO(text.encode("utf-8")))
    if frame.empty:
        raise ValueError("CSV 沒有資料列。")
    limited = frame.head(max_rows).fillna("")
    lines = [f"欄位：{', '.join(map(str, limited.columns))}"]
    for index, row in limited.iterrows():
        fields = [f"{column}={row[column]}" for column in limited.columns]
        lines.append(f"第 {index + 1} 列：" + "；".join(fields))
    if len(frame) > max_rows:
        lines.append(f"[僅載入前 {max_rows} 列；原檔共有 {len(frame)} 列]")
    return "\n".join(lines)


csv_bytes = "name,score\nAlice,90\nBob,85".encode("utf-8")
print(extract_csv_text(csv_bytes))

In [ ]:
def extract_text(filename: str, data: bytes) -> str:
    """依副檔名路由到正確 reader，回傳統一的文字字串。"""
    if "." not in filename:
        raise ValueError("檔名沒有副檔名，無法判斷格式。")
    suffix = filename.lower().rsplit(".", maxsplit=1)[-1]
    readers = {
        "txt": decode_text_bytes,
        "md": decode_text_bytes,
        "pdf": extract_pdf_text,
        "docx": extract_docx_text,
        "csv": extract_csv_text,
    }
    if suffix not in readers:
        raise ValueError(f"不支援 .{suffix}；請使用 PDF、DOCX、CSV、TXT 或 MD。")
    return readers[suffix](data)


print(extract_text("sample.md", b"# Week 9\nDocument pipeline"))

In [ ]:
def clean_text(text: str) -> str:
    """統一換行、移除行內多餘空白，並保留段落分隔。"""
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in normalized.split("\n")]
    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


messy = "標題   有多餘空白\r\n\r\n\r\n第二段\t內容"
print(repr(clean_text(messy)))

In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 120) -> list[dict]:
    """用滑動視窗切分文字，回傳含 chunk_id 與字元位置的資料。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必須大於 0。")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap 必須介於 0（含）與 chunk_size（不含）之間。")

    chunks: list[dict] = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        content = text[start:end].strip()
        if content:
            chunks.append({
                "chunk_id": len(chunks),
                "start": start,
                "end": end,
                "text": content,
            })
        if end == len(text):
            break
        start = end - overlap
    return chunks


print("chunk_text 已定義")

In [ ]:
demo_text = clean_text(
    "第一段介紹文件讀取。" * 12
    + "\n\n"
    + "第二段介紹文字清理。" * 12
    + "\n\n"
    + "第三段介紹 chunking。" * 12
)
demo_chunks = chunk_text(demo_text, chunk_size=120, overlap=20)

print(f"demo_text 長度 = {len(demo_text)}；chunk 數 = {len(demo_chunks)}\n")
for chunk in demo_chunks:
    print(f"chunk={chunk['chunk_id']} range={chunk['start']}:{chunk['end']} length={len(chunk['text'])}")
    print(chunk["text"][:40], "...\n")

In [ ]:
def run_local_checks() -> None:
    """不呼叫 API，快速檢查清理、路由與 chunking 的關鍵邊界。"""
    assert clean_text("A   B\n\n\nC") == "A B\n\nC"
    assert extract_text("note.txt", "測試".encode("utf-8")) == "測試"
    assert len(chunk_text("A" * 250, chunk_size=100, overlap=20)) == 3

    for bad in ({"chunk_size": 0, "overlap": 0}, {"chunk_size": 100, "overlap": 100}):
        try:
            chunk_text("測試", **bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"應拒絕不合法設定：{bad}")

    print("✅ 本機檢查通過：未呼叫付費 API")


run_local_checks()

In [ ]:
from openai import OpenAI


def summarize_document(text: str, model: str | None = None) -> str:
    """用 Responses API 摘要已抽取文字；呼叫前先限制輸入長度。"""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY；請先設定環境變數或 Colab Secret。")

    client = OpenAI(api_key=api_key)
    selected_model = model or os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
    response = client.responses.create(
        model=selected_model,
        instructions=(
            "你是嚴謹的文件整理助理。只能根據文件內容回答；"
            "若文字擷取不完整，必須明確說明。"
        ),
        input=f"請用繁體中文整理摘要、5 個重點與資料限制：\n\n{text[:12000]}",
    )
    if not response.output_text:
        raise RuntimeError("模型沒有回傳文字結果。")
    return response.output_text


print("summarize_document 已定義")

In [ ]:
# 付費 API 預設關閉。確認已設定測試用 API key 後，再改成 True。
RUN_PAID_API = False

if RUN_PAID_API:
    print(summarize_document(demo_text))
else:
    print("已略過付費 API；本機文件處理功能仍可完整練習。")

In [ ]:
def chunk_by_paragraph(text: str, chunk_size: int = 800) -> list[dict]:
    """教師參考：優先合併完整段落，過長段落再用固定長度切割。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必須大於 0。")

    paragraphs = [part.strip() for part in text.split("\n\n") if part.strip()]
    chunks: list[dict] = []
    buffer = ""

    def append_chunk(content: str) -> None:
        chunks.append({"chunk_id": len(chunks), "text": content})

    for paragraph in paragraphs:
        if len(paragraph) > chunk_size:
            if buffer:
                append_chunk(buffer)
                buffer = ""
            for start in range(0, len(paragraph), chunk_size):
                append_chunk(paragraph[start:start + chunk_size])
        elif not buffer:
            buffer = paragraph
        elif len(buffer) + 2 + len(paragraph) <= chunk_size:
            buffer += "\n\n" + paragraph
        else:
            append_chunk(buffer)
            buffer = paragraph

    if buffer:
        append_chunk(buffer)
    return chunks


paragraph_chunks = chunk_by_paragraph(demo_text, chunk_size=160)
print([(item["chunk_id"], len(item["text"])) for item in paragraph_chunks])

In [ ]:
def build_document_report(filename: str, cleaned_text: str, chunks: list[dict]) -> dict:
    """教師參考：建立可記錄、可測試的文件處理摘要。"""
    lengths = [len(item["text"]) for item in chunks]
    return {
        "filename": filename,
        "character_count": len(cleaned_text),
        "chunk_count": len(chunks),
        "average_chunk_length": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "possible_ocr_needed": filename.lower().endswith(".pdf") and not cleaned_text.strip(),
    }


report = build_document_report("demo.txt", demo_text, demo_chunks)
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
challenge_plan = {
    "feature": "只摘要目前選取的 chunk",
    "input": "使用者在 Chunks 分頁選到的 chunk 文字",
    "output": "針對該 chunk 的摘要與重點",
    "error_cases": ["沒有上傳檔案", "chunks 為空", "沒有 API key"],
    "manual_test": "選第 0 塊與最後一塊各摘要一次，確認結果不同且不報錯",
}
print(json.dumps(challenge_plan, ensure_ascii=False, indent=2))